# 🚀 GRPO Training: Teaching AI to Write Optimized Code

## The Challenge

Can we teach an LLM to generate fast, correct Python code through reinforcement learning?

**The Task:**  
Generate optimized matrix multiplication functions using **only native Python** (no NumPy, no external libraries).

**The Method:**  
GRPO (Group Relative Policy Optimization) with custom reward functions.

**What you'll learn:**
- How to set up GRPO training for code generation
- Designing reward functions for RL
- Real-world challenges in AI training

Let's dive in... 🔍

---

## 📚 Background: GRPO for Code Generation

**GRPO** is a reinforcement learning technique that:
- Generates multiple completions per prompt
- Ranks them relative to each other
- Updates the model to favor better solutions

**Why it's interesting:**  
Unlike supervised fine-tuning, the model learns from trial and error.

---

## 🎯 Our Goal

Train Qwen2.5-Coder-7B to write Python functions that:
- ✅ Are syntactically valid
- ✅ Solve the problem correctly
- ✅ Run as fast as possible (pure Python)

Let's see how it goes...

---

## 📦 Part 1: Environment Setup

Installing the required packages for GRPO training.

**Key dependencies:**
- **unsloth** - Efficient LLM fine-tuning library
- **trl** - Transformers Reinforcement Learning (includes GRPOTrainer)
- **transformers** - Hugging Face model library
- **torch** - PyTorch for GPU acceleration

⏱️ **Installation time:** ~2-3 minutes

In [2]:
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):    
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2

Using Python 3.11.13 environment at: /usr
Resolved 3 packages in 50ms                                          
Audited 3 packages in 0.29ms


In [3]:

try:
    import torch
    import transformers
    import trl
    import unsloth
    print("✅ torch OK")
    print("✅ transformers OK")
    print("✅ trl OK")
    print("✅ unsloth OK")
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n⚠️ Réinstallez avec:")
    print("!pip install --force-reinstall unsloth")

/tmp/ipykernel_383/1176904608.py:5: UserWarning: WARNING: Unsloth should be imported before trl, transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-10-17 23:41:05.165140: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760744465.192367     383 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760744465.200325     383 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `ty

🦥 Unsloth Zoo will now patch everything to make training faster!
✅ torch OK
✅ transformers OK
✅ trl OK
✅ unsloth OK


---

## 🤖 Part 2: Load the Base Model

We're using **Qwen2.5-Coder-7B-Instruct** with:
- 4-bit quantization (to fit in GPU memory)
- LoRA adapters (parameter-efficient fine-tuning)
- Gradient checkpointing (memory optimization)

**Model specs:**
- Target modules: Q, K, V, O projections
- LoRA rank: 8
- Trainable params: ~5M out of 7.6B (0.07%)

In [4]:
import os
import gc
import torch
import numpy as np
import time
import statistics
import signal
from contextlib import contextmanager
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from transformers import TextStreamer

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

_original_argsort = torch.argsort
def patched_argsort(input, dim=-1, descending=False, stable=False):
    if input.dtype == torch.bool:
        input = input.to(torch.int32)
    return _original_argsort(input, dim=dim, descending=descending, stable=stable)
torch.argsort = patched_argsort

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
    device_map={"": 0},
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id

print("✅ Modèle chargé")

==((====))==  Unsloth 2025.10.5: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2025.10.5 patched 28 layers with 28 QKV layers, 28 O layers and 0 MLP layers.


✅ Modèle chargé


---

## ⚙️ Part 3: Utilities & Reward Functions

### Reward Design

We implement TWO reward functions:

#### 1. `minimal_check` 🔍
Verifies that generated code is syntactically valid Python and can be compiled.

#### 2. `speed_check_naive` ⏱️
Benchmarks the generated function and compares its execution speed to NumPy's implementation.

---

### Helper Functions
- `generate_random_matrices()`: Creates test matrices for benchmarking
- `extract_function()`: Parses code from model output
- `Benchmarker`: Measures execution time with cache management

In [5]:
def generate_random_matrices(seed=3407, n=32):
    random_state = np.random.RandomState(seed)
    n, k, m = random_state.randint(1, n+1, size=3)
    A = np.random.uniform(-10, 10, size=(n, k)).astype(np.float32)
    B = np.random.uniform(-10, 10, size=(k, m)).astype(np.float32)
    return A, A.tolist(), B, B.tolist()

def extract_function(text):
    if text.count("```") >= 2:
        first = text.find("```") + 3
        second = text.find("```", first)
        fx = text[first:second].strip()
        fx = fx.removeprefix("python\n")
        fx = fx[fx.find("def"):] if "def" in fx else None
        if fx and fx.startswith("def matmul(A, B):"):
            return fx
    return None

class TimeoutError(Exception): 
    pass

@contextmanager
def time_limit(seconds):
    def _handler(signum, frame):
        raise TimeoutError(f"Timed out after {seconds}s")
    old = signal.signal(signal.SIGALRM, _handler)
    signal.setitimer(signal.ITIMER_REAL, seconds)
    try:
        yield
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, old)

class Benchmarker:
    def __init__(self, trials=1, loops=1, timeout=3):
        self.buffer = np.zeros(128 * 1024 * 1024, dtype=np.uint8)
        self.trials = trials
        self.loops = loops
        self.timeout = timeout
        
    def thrash(self):
        self.buffer ^= 1
        return int(self.buffer[::16384].sum())
    
    def benchmark(self, function, arguments):
        samples = []
        for _ in range(self.trials):
            gc.collect()
            self.thrash()
            t_start = time.perf_counter_ns()
            try:
                with time_limit(self.timeout):
                    function(*arguments[0])
            except:
                pass
            t_end = time.perf_counter_ns()
            samples.append(t_end - t_start)
        return {"median_ns": int(statistics.median(samples)) if samples else 1000000}

prompt = """Create a new fast matrix multiplication function using only native Python code.
You are given a list of list of numbers.
Output your new function in backticks using the format below:
````python
def matmul(A, B):
    return ...
```""".strip()

maximum_length = len(tokenizer(prompt.strip())["input_ids"])

print("✅ Utilitaires OK")
def speed_check_naive(completions, **kwargs):
    scores = []
    A, A_list, B, B_list = generate_random_matrices(seed=np.random.randint(10000), n=32)
    benchmarker = Benchmarker(trials=1, timeout=3)
    
    try:
        numpy_results = benchmarker.benchmark(np.matmul, [(A, B)])
    except:
        numpy_results = {"median_ns": 1000000}
    
    for completion in completions:
        response = completion[0]["content"]
        function = extract_function(response)
        
        if function is None:
            scores.append(-10.0)
            continue
            
        try:
            exec(function, globals())
            new_matmul = globals()["matmul"]
            new_results = benchmarker.benchmark(new_matmul, [(A_list, B_list)])
            ratio = numpy_results["median_ns"] / max(new_results["median_ns"], 1)
            score = min(ratio * 10, 100)
            scores.append(score)
        except:
            scores.append(-5.0)
    
    del A, B, A_list, B_list
    gc.collect()
    torch.cuda.empty_cache()
    return scores

def minimal_check(completions, **kwargs):
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        function = extract_function(response)
        
        if function is None:
            scores.append(-2.0)
        else:
            try:
                compile(function, '<string>', 'exec')
                scores.append(1.0)
            except:
                scores.append(-2.0)
    return scores

print("✅ Reward functions créées (NAIVE - SANS PROTECTIONS)")


✅ Utilitaires OK
✅ Reward functions créées (NAIVE - SANS PROTECTIONS)


---

## 🎓 Part 4: Training Configuration

### Dataset Preparation

We create a simple dataset with:
- 30 identical training samples (same prompt repeated)
- The prompt asks for a fast matrix multiplication function
- This repetition helps the model learn the pattern quickly

### GRPO Configuration

**GRPO (Group Relative Policy Optimization)** is a reinforcement learning algorithm that:
- Generates multiple completions per prompt
- Compares them relatively to each other
- Updates the model to favor higher-reward outputs

**Key hyperparameters:**
- `max_steps`: 15 (short training for experimentation)
- `num_generations`: 2 (generate 2 solutions per prompt to compare)
- `per_device_train_batch_size`: 2 (adjusted automatically)
- `gradient_accumulation_steps`: 4 (effective batch size = 8)
- `temperature`: 0.9 (encourages some exploration)
- `learning_rate`: 2e-5 (conservative for LoRA)

### What to Expect

The model will:
1. Generate code solutions
2. Get evaluated by our reward functions
3. Learn which patterns lead to higher rewards
4. Gradually optimize its generation strategy

In [6]:
dataset_naive = Dataset.from_list([{
    "prompt": [{"role": "user", "content": prompt.strip()}], 
    "answer": 0, 
    "reasoning_effort": "low"
}] * 30)

training_args_naive = GRPOConfig(
    temperature=0.9,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=2,
    max_prompt_length=maximum_length + 1,
    max_completion_length=128,
    max_steps=15,
    save_steps=15,
    report_to="none",
    output_dir="outputs_naive",
    fp16=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    torch_compile=False,
    ddp_find_unused_parameters=False,
)

print(f"✅ Config: {len(dataset_naive)} samples, {training_args_naive.max_steps} steps")

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 2
✅ Config: 30 samples, 15 steps


In [7]:
torch.cuda.empty_cache()
gc.collect()

allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"✅ Mémoire: {allocated:.2f}GB alloués / {total:.2f}GB total")

✅ Mémoire: 5.20GB alloués / 14.74GB total


In [8]:
trainer_naive = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[minimal_check, speed_check_naive],
    args=training_args_naive,
    train_dataset=dataset_naive,
)

print("✅ Trainer créé - Prêt pour l'entraînement")

✅ Trainer créé - Prêt pour l'entraînement


---

## 🚀 Part 5: Training

### Starting the Training Process

This is where the magic happens! The model will:
1. Generate multiple code solutions for each prompt
2. Evaluate them using our reward functions
3. Learn from the feedback
4. Update its parameters to improve future generations

### 📊 Metrics to Monitor

During training, watch these key metrics in the progress bar:

**Reward Metrics:**
- `reward`: Combined total reward (average across both functions)
- `reward_std`: Standard deviation (high = varied solutions)
- `rewards/minimal_check/mean`: Compilation success rate
- `rewards/speed_check_naive/mean`: Speed optimization score

**Generation Metrics:**
- `completions/mean_length`: Average tokens generated
- `completions/clipped_ratio`: How often we hit max length
- `kl`: KL divergence from base model (measures how much we've changed)

**Training Metrics:**
- `Training Loss`: Policy gradient loss
- Progress through 15 steps

---

### ⏱️ Expected Duration

With our configuration:
- 15 steps × ~35-40 seconds per step
- Total: ~8-10 minutes on T4 GPU

Let's start training and see what the model learns!

In [9]:


print("🚀 Démarrage de l'entraînement NAIVE (sans protections)...")
print("="*60)

try:
    trainer_naive.train()
    print("\n✅ Entraînement terminé!")
    
    model.save_pretrained("matmul_naive_model")
    tokenizer.save_pretrained("matmul_naive_model")
    print("✅ Modèle sauvegardé")
    
except Exception as e:
    print(f"\n❌ Erreur: {e}")
    import traceback
    traceback.print_exc()
    
finally:
    torch.cuda.empty_cache()
    gc.collect()
    print("\n✅ Nettoyage terminé")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 Démarrage de l'entraînement NAIVE (sans protections)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30 | Num Epochs = 3 | Total steps = 15
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 5,046,272 of 7,620,662,784 (0.07% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / minimal_check / mean,rewards / minimal_check / std,rewards / speed_check_naive / mean,rewards / speed_check_naive / std
1,0.000000,58.625000,19.975767,103.250000,81.000000,128.000000,0.375000,88.400002,81.000000,116.000000,0.000006,-0.125000,1.552648,58.750000,56.930408
2,0.000000,-2.369046,8.168854,104.625000,81.000000,128.000000,0.500000,81.250000,81.000000,82.000000,0.000006,-0.125000,1.552648,-2.244046,6.424469
3,0.000000,-0.832467,9.880142,99.125000,45.000000,128.000000,0.375000,81.800003,45.000000,121.000000,0.000017,-0.125000,1.552648,-0.707467,7.713172
4,0.000000,-2.972143,7.679472,105.875000,50.000000,128.000000,0.500000,83.750000,50.000000,118.000000,0.000334,-0.125000,1.552648,-2.847143,5.924549
5,0.000000,44.500000,79.903069,104.875000,81.000000,128.000000,0.500000,81.750000,81.000000,84.000000,0.001722,-0.500000,1.603567,45.000000,58.797474
6,0.000000,-4.287633,10.906934,96.125000,45.000000,128.000000,0.500000,64.250000,45.000000,81.000000,0.004717,-0.500000,1.603567,-3.787633,6.648367
7,0.000000,-3.427722,6.057145,104.500000,81.000000,128.000000,0.500000,81.000000,81.000000,81.000000,0.007398,-0.500000,1.603567,-2.927722,7.561276
8,0.000100,0.038223,3.027945,103.125000,81.000000,128.000000,0.375000,88.200005,81.000000,117.000000,0.013480,-0.125000,1.552648,0.163223,8.498859
9,0.000100,-0.908005,0.200088,100.250000,81.000000,128.000000,0.250000,91.000000,81.000000,126.000000,0.019899,0.250000,1.388730,-1.158005,5.461832
10,0.000000,-10.057338,2.747340,122.000000,80.000000,128.000000,0.875000,80.000000,80.000000,80.000000,0.004623,-1.625000,1.060660,-8.432338,4.434019



✅ Entraînement terminé!
✅ Modèle sauvegardé

✅ Nettoyage terminé


---

## 🧪 Part 6: Testing The Trained Model

### Evaluation Time

Now that training is complete, let's test what the model has learned!

We'll:
1. Put the model in evaluation mode
2. Give it the same prompt we used during training
3. Generate a new matrix multiplication function
4. Extract and display the result

### Generation Parameters

- `temperature`: 0.8 (slightly creative but focused)
- `max_new_tokens`: 150 (enough for a complete function)
- `do_sample`: True (allow for diverse outputs)

### What Will We Get?

The model has been trained with our reward function

In [10]:
model.eval()

print("🧪 Test du nouveau modèle\n")

text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        temperature=0.8,
        max_new_tokens=150,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
function = extract_function(generated)

print("Résultat:")
print("-"*60)
print(function if function else "❌ Pas de fonction extraite")
print("-"*60)

🧪 Test du nouveau modèle

Résultat:
------------------------------------------------------------
def matmul(A, B):
    return ...
------------------------------------------------------------


---

## 📊 Results & Analysis

### What Happened?

Look at the generated function above. Ask yourself:
- Does it actually solve matrix multiplication?
- What strategy did the model learn?
- Why did the training metrics evolve this way?

---

## 💡 Key Takeaway

**RL models optimize for what you measure, not what you intend.**

This experiment shows why reward function design is critical:
- Simple metrics (like "does it compile?") can be exploited
- Code generation needs correctness checks, not just syntax validation
- Unexpected behaviors emerge when incentives misalign

---

## 🔧 What's Next?

To improve this approach:
- Add correctness verification with test cases
- Use isolated execution environments
- Penalize suspicious patterns
- Randomize test data for each evaluation

---

## 📚 Resources

- [GRPO Paper](https://arxiv.org/abs/2402.03300)
- [Unsloth](https://github.com/unslothai/unsloth)
- [TRL Docs](https://huggingface.co/docs/trl)

---

**Experiment completed!** 🎉

#MachineLearning #RLHF #GRPO #CodeGeneration #AIAlignment